# Partitioned LDSC

Goal: run partitioned LDSC in the refactored codebase by building query annotations, computing baseline-plus-query LD scores from a canonical parquet R2 reference panel, and regressing one trait on that partitioned design.

The parquet R2 file should use the canonical six-column schema (`CHR`, `POS_1`, `POS_2`, `SNP_1`, `SNP_2`, `R2`) with row-group statistics. The paired metadata sidecar is required and defines the raw reference-panel SNP universe; parquet pair rows are queried only for LD values. Query annotations require explicit baseline annotations; the synthetic all-ones `base` path is only for `h2`/`rg` runs with no query inputs. `partitioned-h2` rejects baseline-only LD-score directories.

Path-token rules used below:
- use `@` for chromosome suites such as `baseline.@.annot.gz`
- use globs when the filenames do not follow the simple chromosome-suffix convention
- scalar inputs still resolve to exactly one file
- output directories are explicit workflow destinations
- missing output directories are created and existing directories are reused
- existing owned workflow artifacts are refused unless you pass `--overwrite` or `overwrite=True`; successful overwrites remove stale siblings from the same family, such as old query annotation shards, `ldscore.query.parquet`, or `query_annotations/`


In [ ]:
import os
import sys

# Set this explicitly for your environment.
SRC_DIR = "directory/to/src"

if not os.path.isdir(SRC_DIR):
    raise FileNotFoundError(f"Set SRC_DIR to an existing path before running the notebook: {SRC_DIR}")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

## Process Annotation Files

In [ ]:
from ldsc import GlobalConfig, get_global_config, run_bed_to_annot, set_global_config

In [ ]:
# Set these explicitly for your environment.
OUTPUT_DIR = "directory/for/outputs"

BED_FILES = "path/to/raw_bed/*.bed"  # use a glob token to include all BED files
BASELINE_ANNOT_SOURCES = "directory/to/baseline_annot/baseline.@.annot.gz"
QUERY_ANNOT_OUTPUT_DIR = "directory/to/processed_query_annot"

# Register shared runtime assumptions once at the top of the workflow.
# Per-run SNP-universe filters belong to RefPanelConfig or LDScoreConfig.
set_global_config(GlobalConfig(
    snp_identifier="chr_pos",
    genome_build="hg38",
))
GLOBAL_CONFIG = get_global_config()

In [ ]:
run_bed_to_annot(
    query_annot_bed_sources=BED_FILES,
    baseline_annot_sources=BASELINE_ANNOT_SOURCES,
    output_dir=QUERY_ANNOT_OUTPUT_DIR,
    # overwrite=True,  # also removes stale query.<chrom>.annot.gz shards outside this run
)

## Calculate LD Scores

In [ ]:
from ldsc import (
    AnnotationBuildConfig,
    AnnotationBuilder,
    GlobalConfig,
    LDScoreCalculator,
    LDScoreConfig,
    LDScoreOutputConfig,
    PartitionedH2DirectoryWriter,
    PartitionedH2OutputConfig,
    RefPanelLoader,
    RefPanelConfig,
    RegressionConfig,
    RegressionRunner,
    get_global_config,
    set_global_config,
)

In [ ]:
# Keep this consistent with the baseline annotation build.
# ref_panel_snps_file restricts the sidecar-defined reference-panel rows and therefore the compute-time universe B ∩ A'.
# regression_snps_file restricts the normalized baseline_table/query_table rows to B ∩ A' ∩ C.
set_global_config(GlobalConfig(
    snp_identifier="chr_pos",
    genome_build="hg19",
))
GLOBAL_CONFIG = get_global_config()

OUTPUT_DIR = "directory/for/outputs"
BASELINE_ANNOT_SOURCES = "path/to/baseline_annot/baseline.@.annot.gz"
QUERY_ANNOT_SOURCES = "path/to/query_annot/query.@.annot.gz"
REF_PANEL_DIR = "path/to/ref_panel/hg19"
R2_BIAS_MODE = "unbiased"
REF_PANEL_SNPS_FILE = "filters/reference_universe.tsv.gz"
REGRESSION_SNPS_FILE = "filters/hapmap3.tsv.gz"
LD_WIND_CM = 1.0


In [ ]:
annotation_bundle = AnnotationBuilder(GLOBAL_CONFIG, AnnotationBuildConfig()).run(
    AnnotationBuildConfig(
        baseline_annot_sources=BASELINE_ANNOT_SOURCES,
        query_annot_sources=QUERY_ANNOT_SOURCES,
    )
)

ref_panel = RefPanelLoader(GLOBAL_CONFIG).load(
    RefPanelConfig(
        backend="parquet_r2",
        r2_dir=REF_PANEL_DIR,
        r2_bias_mode=R2_BIAS_MODE,
        chromosomes=tuple(annotation_bundle.chromosomes),
        ref_panel_snps_file=REF_PANEL_SNPS_FILE,
    )
)

ldscore_result = LDScoreCalculator().run(
    annotation_bundle=annotation_bundle,
    ref_panel=ref_panel,
    ldscore_config=LDScoreConfig(
        ld_wind_cm=LD_WIND_CM,
        regression_snps_file=REGRESSION_SNPS_FILE,
    ),
    global_config=GLOBAL_CONFIG,
    output_config=LDScoreOutputConfig(
        output_dir=OUTPUT_DIR,
        # overwrite=True,  # also removes stale LD-score siblings not produced by this run
    ),
)

runner = RegressionRunner(global_config=GLOBAL_CONFIG, regression_config=RegressionConfig())

Since the annotation bundle, restricted reference panel, and optional regression SNP list do not cover exactly the same SNP universe, the LD-score workflow uses two nested intersections:

- `ld_reference_snps = B ∩ A'`, where `B` is the annotation bundle and `A'` is the sidecar-defined reference panel after `ref_panel_snps_file`
- `ld_regression_snps = B ∩ A' ∩ C`, where `C` comes from `regression_snps_file`
- count records are computed over `ld_reference_snps`, while `baseline_table` and optional `query_table` rows are restricted to `ld_regression_snps`


In [ ]:
before = annotation_bundle.metadata["CHR"].value_counts().sort_index()
after = ldscore_result.baseline_table["CHR"].value_counts().sort_index()

comparison = (
    before.rename("annotation_bundle")
    .to_frame()
    .join(after.rename("retained_in_ldscore_rows"), how="outer")
    .fillna(0)
    .astype(int)
)
comparison["dropped"] = comparison["annotation_bundle"] - comparison["retained_in_ldscore_rows"]
comparison

## Estimate Partitioned Heritability

In [ ]:
from ldsc import load_sumstats

In [ ]:
SUMSTATS_FILE = "path/to/normalized_sumstats/sumstats.parquet"
TRAIT_NAME = "trait"
PARTITIONED_OUTPUT_DIR = f"{OUTPUT_DIR}/pldsc_results/partitioned_h2"
PARTITIONED_OUTPUT_PATH = f"{PARTITIONED_OUTPUT_DIR}/partitioned_h2.tsv"
PARTITIONED_LOG_PATH = f"{PARTITIONED_OUTPUT_DIR}/partitioned-h2.log"

In [ ]:
# Current parquet and legacy .sumstats.gz artifacts recover config_snapshot
# from sumstats.metadata.json. Older files without that sidecar warn and use
# config_snapshot=None. Keep GLOBAL_CONFIG aligned with legacy artifacts
# before loading.
sumstats = load_sumstats(
    SUMSTATS_FILE,
    trait_name=TRAIT_NAME,
)

partitioned_result = runner.estimate_partitioned_h2_batch(
    sumstats,
    ldscore_result,
    annotation_bundle,
    include_full_partitioned_h2=True,
)
partitioned = partitioned_result.summary

PartitionedH2DirectoryWriter().write(
    partitioned,
    PartitionedH2OutputConfig(
        output_dir=PARTITIONED_OUTPUT_DIR,
        overwrite=True,
        write_per_query_results=True,
    ),
    per_query_category_tables=partitioned_result.per_query_category_tables,
    metadata={"trait_name": TRAIT_NAME, "count_kind": "common", "ldscore_dir": str(LDSCORE_OUTPUT_DIR)},
    per_query_metadata=partitioned_result.per_query_metadata,
)
partitioned